# Spatial Analysis: WBC Clustering (Objective 3)

This notebook implements Objective 3: spatial analysis of false positive distribution.

**Hypothesis**: False positives cluster around white blood cell (WBC) centers.

**Method**:
1. Extract FP and TP prediction centers from Model A
2. Extract annotated WBC centers from Model B predictions
3. Use DBSCAN to identify clusters of FPs around WBCs
4. Compute distance-to-nearest-WBC for FPs vs TPs
5. Perform Mann-Whitney U test to determine statistical significance

In [ ]:
# Cell 1: Setup
import os
from pathlib import Path
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    project_root = Path("/content/drive/MyDrive/new_spark_capstone")
else:
    project_root = Path.cwd()

# Paths to FP analysis output and models
predictions_file = project_root / "predictions_with_tp_fp_labels.csv"
model_a = project_root / "model_checkpoints" / "model_a_best.pt"
model_b = project_root / "model_checkpoints" / "model_b_best.pt"
dataset_root = project_root / "Spark_Capstone.v2-augmented.yolov8"

print(f"Predictions file: {predictions_file.exists()}")
print(f"Model A: {model_a.exists()}")
print(f"Model B: {model_b.exists()}")

In [ ]:
# Cell 2: Install packages
import subprocess
import sys

packages = ['ultralytics', 'pandas', 'numpy', 'scipy', 'scikit-learn']
for pkg in packages:
    print(f"Installing {pkg}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("All packages installed!")

In [ ]:
# Cell 3: Load FP predictions from previous analysis
import pandas as pd

df_predictions = pd.read_csv(predictions_file)

print(f"Loaded {len(df_predictions)} predictions")
print(f"\nTP/FP breakdown:")
print(df_predictions[['is_tp', 'is_fp']].value_counts())

# Extract FP and TP centers
fp_predictions = df_predictions[df_predictions['is_fp']].copy()
tp_predictions = df_predictions[df_predictions['is_tp']].copy()

print(f"\nFalse Positives: {len(fp_predictions)}")
print(f"True Positives: {len(tp_predictions)}")

In [ ]:
# Cell 4: Extract WBC centers from Model B predictions
# NOTE: This requires Model B to have been trained with WBC annotations
# If WBC annotations are not available, this cell will show empty WBC detections

from ultralytics import YOLO
from pathlib import Path

# Load Model B (two-class detector)
model_b_obj = YOLO(str(model_b))

test_images_dir = dataset_root / "test" / "images"
print(f"Running Model B inference to extract WBC centers...")

results = model_b_obj.predict(
    source=str(test_images_dir),
    save=False,
    conf=0.3,
    verbose=False
)

# Extract WBC centers (class 1)
wbc_data = []
for r in results:
    image_id = Path(r.path).stem
    for box in r.boxes:
        class_id = int(box.cls[0])
        if class_id == 1:  # WBC class
            wbc_data.append({
                'image_id': image_id,
                'x_center': float(box.xywh[0][0]),
                'y_center': float(box.xywh[0][1]),
                'confidence': float(box.conf[0])
            })

df_wbc = pd.DataFrame(wbc_data)
print(f"\nExtracted {len(df_wbc)} WBC detections")
print(f"WBCs detected in {df_wbc['image_id'].nunique()} images")

if len(df_wbc) == 0:
    print("\nWARNING: No WBCs detected. Model B may not have been trained with WBC annotations.")

In [ ]:
# Cell 5: Compute distance from FPs/TPs to nearest WBC
import numpy as np
from scipy.spatial import cKDTree

def compute_distances_to_nearest_wbc(predictions_df, wbc_df):
    """
    For each prediction, find the distance to the nearest WBC in the same image.
    """
    distances = []
    
    for img_id in predictions_df['image_id'].unique():
        pred_in_img = predictions_df[predictions_df['image_id'] == img_id]
        wbc_in_img = wbc_df[wbc_df['image_id'] == img_id]
        
        if len(wbc_in_img) == 0:
            # No WBCs in this image; set distance to NaN
            for _, pred in pred_in_img.iterrows():
                distances.append({
                    'image_id': img_id,
                    'pred_x': pred['x_center'],
                    'pred_y': pred['y_center'],
                    'distance_to_nearest_wbc': np.nan
                })
        else:
            # Build KDTree for WBCs in this image
            wbc_coords = wbc_in_img[['x_center', 'y_center']].values
            tree = cKDTree(wbc_coords)
            
            # Query distance for each prediction
            pred_coords = pred_in_img[['x_center', 'y_center']].values
            dists, _ = tree.query(pred_coords)
            
            for i, (_, pred) in enumerate(pred_in_img.iterrows()):
                distances.append({
                    'image_id': img_id,
                    'pred_x': pred['x_center'],
                    'pred_y': pred['y_center'],
                    'distance_to_nearest_wbc': dists[i]
                })
    
    return pd.DataFrame(distances)

if len(df_wbc) > 0:
    print("Computing distances to nearest WBC...")
    
    fp_distances = compute_distances_to_nearest_wbc(fp_predictions, df_wbc)
    tp_distances = compute_distances_to_nearest_wbc(tp_predictions, df_wbc)
    
    print(f"\nFP Distance Statistics (to nearest WBC):")
    print(f"  Valid samples: {fp_distances['distance_to_nearest_wbc'].notna().sum()}")
    print(f"  Mean: {fp_distances['distance_to_nearest_wbc'].mean():.2f} px")
    print(f"  Median: {fp_distances['distance_to_nearest_wbc'].median():.2f} px")
    print(f"  Std: {fp_distances['distance_to_nearest_wbc'].std():.2f} px")
    print(f"  Min: {fp_distances['distance_to_nearest_wbc'].min():.2f} px")
    print(f"  Max: {fp_distances['distance_to_nearest_wbc'].max():.2f} px")
    
    print(f"\nTP Distance Statistics (to nearest WBC):")
    print(f"  Valid samples: {tp_distances['distance_to_nearest_wbc'].notna().sum()}")
    print(f"  Mean: {tp_distances['distance_to_nearest_wbc'].mean():.2f} px")
    print(f"  Median: {tp_distances['distance_to_nearest_wbc'].median():.2f} px")
    print(f"  Std: {tp_distances['distance_to_nearest_wbc'].std():.2f} px")
    print(f"  Min: {tp_distances['distance_to_nearest_wbc'].min():.2f} px")
    print(f"  Max: {tp_distances['distance_to_nearest_wbc'].max():.2f} px")
else:
    print("Skipping distance analysis: no WBCs detected")

In [ ]:
# Cell 6: Mann-Whitney U test
from scipy.stats import mannwhitneyu
import pandas as pd

if len(df_wbc) > 0:
    # Filter out NaN distances
    fp_dists_valid = fp_distances['distance_to_nearest_wbc'].dropna().values
    tp_dists_valid = tp_distances['distance_to_nearest_wbc'].dropna().values
    
    if len(fp_dists_valid) > 0 and len(tp_dists_valid) > 0:
        # Mann-Whitney U test (one-tailed: FPs are closer to WBCs)
        stat, p_value = mannwhitneyu(fp_dists_valid, tp_dists_valid, alternative='less')
        
        print("\n" + "="*60)
        print("Mann-Whitney U Test Results")
        print("="*60)
        print(f"Hypothesis: FPs cluster closer to WBCs than TPs")
        print(f"\nU-statistic: {stat:.2f}")
        print(f"P-value (one-tailed, 'less'): {p_value:.6f}")
        
        if p_value < 0.05:
            print(f"\n*** SIGNIFICANT (p < 0.05) ***")
            print(f"Evidence: FPs are significantly closer to WBCs than TPs.")
        else:
            print(f"\n*** NOT SIGNIFICANT (p >= 0.05) ***")
            print(f"No strong evidence that FPs cluster around WBCs.")
    else:
        print("Insufficient valid distance data for Mann-Whitney test")
else:
    print("Skipping Mann-Whitney test: no WBCs detected")

In [ ]:
# Cell 7: DBSCAN clustering analysis
from sklearn.cluster import DBSCAN
import numpy as np

if len(df_wbc) > 0 and len(fp_predictions) > 0:
    print("\n" + "="*60)
    print("DBSCAN Clustering Analysis")
    print("="*60)
    
    # Cluster FP predictions
    fp_coords = fp_predictions[['x_center', 'y_center']].values
    
    # DBSCAN parameters
    eps = 50  # Radius in pixels
    min_samples = 2  # Minimum samples to form a cluster
    
    clustering = DBSCAN(eps=eps, min_samples=min_samples).fit(fp_coords)
    labels = clustering.labels_
    
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = list(labels).count(-1)
    
    print(f"\nDBSCAN Parameters:")
    print(f"  eps (radius): {eps} px")
    print(f"  min_samples: {min_samples}")
    
    print(f"\nClustering Results:")
    print(f"  Number of clusters: {n_clusters}")
    print(f"  Number of noise points: {n_noise}")
    print(f"  Clustered points: {len(labels) - n_noise}")
    print(f"  Clustering ratio: {(len(labels) - n_noise) / len(labels):.2%}")
    
    # Analyze cluster-WBC proximity
    print(f"\nAnalyzing cluster proximity to WBCs...")
    
    wbc_coords = df_wbc[['x_center', 'y_center']].values
    from scipy.spatial import cKDTree
    wbc_tree = cKDTree(wbc_coords)
    
    for cluster_id in range(n_clusters):
        cluster_mask = labels == cluster_id
        cluster_points = fp_coords[cluster_mask]
        
        # Distance from cluster center to nearest WBC
        cluster_center = cluster_points.mean(axis=0)
        dist_to_wbc, _ = wbc_tree.query([cluster_center])
        
        print(f"  Cluster {cluster_id}: {cluster_points.shape[0]} points, " 
              f"nearest WBC at {dist_to_wbc[0]:.1f} px")
else:
    print("Skipping DBSCAN analysis: insufficient FPs or WBCs")

In [ ]:
# Cell 8: Visualization
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: Distance distributions
if len(df_wbc) > 0:
    ax = axes[0]
    fp_dists = fp_distances['distance_to_nearest_wbc'].dropna().values
    tp_dists = tp_distances['distance_to_nearest_wbc'].dropna().values
    
    if len(fp_dists) > 0 and len(tp_dists) > 0:
        ax.hist(fp_dists, bins=30, alpha=0.6, label=f'FP (n={len(fp_dists)})', color='red')
        ax.hist(tp_dists, bins=30, alpha=0.6, label=f'TP (n={len(tp_dists)})', color='green')
        ax.set_xlabel('Distance to Nearest WBC (pixels)')
        ax.set_ylabel('Frequency')
        ax.set_title('Distance to Nearest WBC: FP vs TP')
        ax.legend()
        ax.grid(True, alpha=0.3)

# Plot 2: Scatter plot of FP/TP/WBC locations (sample image)
ax = axes[1]
if len(df_wbc) > 0 and len(fp_predictions) > 0:
    # Plot first image with predictions and WBCs
    sample_image_id = fp_predictions['image_id'].iloc[0]
    
    fps_sample = fp_predictions[fp_predictions['image_id'] == sample_image_id]
    tps_sample = tp_predictions[tp_predictions['image_id'] == sample_image_id]
    wbcs_sample = df_wbc[df_wbc['image_id'] == sample_image_id]
    
    if len(fps_sample) > 0:
        ax.scatter(fps_sample['x_center'], fps_sample['y_center'], 
                  s=100, alpha=0.6, color='red', marker='x', label=f'FP (n={len(fps_sample)})')
    if len(tps_sample) > 0:
        ax.scatter(tps_sample['x_center'], tps_sample['y_center'], 
                  s=100, alpha=0.6, color='green', marker='o', label=f'TP (n={len(tps_sample)})')
    if len(wbcs_sample) > 0:
        ax.scatter(wbcs_sample['x_center'], wbcs_sample['y_center'], 
                  s=150, alpha=0.6, color='blue', marker='^', label=f'WBC (n={len(wbcs_sample)})')
    
    ax.set_xlim([0, 640])
    ax.set_ylim([640, 0])
    ax.set_aspect('equal')
    ax.set_xlabel('X (pixels)')
    ax.set_ylabel('Y (pixels)')
    ax.set_title(f'Sample Image: {sample_image_id}')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(project_root / 'spatial_analysis_visualization.png', dpi=100)
plt.show()

print("Visualization saved to: spatial_analysis_visualization.png")

In [ ]:
# Cell 9: Export spatial analysis results
if len(df_wbc) > 0:
    # Export distance data
    fp_distances.to_csv(project_root / 'fp_distances_to_wbc.csv', index=False)
    tp_distances.to_csv(project_root / 'tp_distances_to_wbc.csv', index=False)
    
    # Create summary report
    summary_stats = pd.DataFrame({
        'metric': ['count', 'mean_distance', 'median_distance', 'std_distance'],
        'fp': [
            len(fp_dists_valid),
            fp_dists_valid.mean(),
            np.median(fp_dists_valid),
            fp_dists_valid.std()
        ],
        'tp': [
            len(tp_dists_valid),
            tp_dists_valid.mean(),
            np.median(tp_dists_valid),
            tp_dists_valid.std()
        ]
    })
    summary_stats.to_csv(project_root / 'spatial_analysis_summary.csv', index=False)
    
    print("Spatial analysis results exported:")
    print(f"  - fp_distances_to_wbc.csv")
    print(f"  - tp_distances_to_wbc.csv")
    print(f"  - spatial_analysis_summary.csv")
else:
    print("No spatial analysis results to export (no WBCs detected)")